# 02 - Clean and Load Synthetic Data

This notebook cleans intentionally messy raw files and exports analyst-ready processed datasets:
- `data/processed/meetings_clean.csv`
- `data/processed/pulse_clean.csv`


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

meetings_path = RAW_DIR / "meetings_raw.csv"
pulse_path = RAW_DIR / "pulse_raw.csv"

meetings_raw = pd.read_csv(meetings_path)
pulse_raw = pd.read_csv(pulse_path)

meetings_raw["date"] = pd.to_datetime(meetings_raw["date"], errors="coerce")
pulse_raw["week"] = pd.to_datetime(pulse_raw["week"], errors="coerce")

print("Loaded raw files:")
print(f"- meetings_raw: {len(meetings_raw):,} rows")
print(f"- pulse_raw   : {len(pulse_raw):,} rows")


## Cleaning decisions (analyst rationale)

- **Duplicates (both tables): remove exact duplicates, keep first** — duplicate logs inflate volume/cost and bias all rate-based metrics.
- **`meetings.duration_mins`: impute missing by meeting-type median** — duration is central to cost, and meeting-type medians preserve business context better than global fill.
- **`meetings.duration_mins <= 0`: treat as invalid and impute** — zero/negative durations are impossible operationally, so they should not survive into analysis.
- **Long meetings (`duration_mins > 240`): cap at 240 and flag** — retain rare workshops while preventing extreme values from dominating cost estimates.
- **`pulse.self_reported_productivity`: impute by (`seniority`, `department`, `hours_bucket`) median** — preserves relationship to meeting load and role context while recovering sparse missing survey values.
- **`pulse.self_reported_burnout`: impute by (`seniority`, `department`, `hours_bucket`) median** — same logic as productivity so burnout trend remains comparable.


In [ ]:
def missing_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2),
        }
    )
    summary = summary[summary["missing_count"] > 0].sort_values("missing_count", ascending=False)
    print(f"\n{name} missing-value summary:")
    if summary.empty:
        print("No missing values.")
    else:
        print(summary)
    return summary


before_counts = {
    "meetings_rows": len(meetings_raw),
    "pulse_rows": len(pulse_raw),
}

print("=== BEFORE CLEANING ===")
print(before_counts)
before_meet_missing = missing_summary(meetings_raw, "meetings_raw")
before_pulse_missing = missing_summary(pulse_raw, "pulse_raw")


## 1) Remove duplicates

In [ ]:
meetings_clean = meetings_raw.copy()
pulse_clean = pulse_raw.copy()

meetings_dupes_removed = int(meetings_clean.duplicated().sum())
pulse_dupes_removed = int(pulse_clean.duplicated().sum())

meetings_clean = meetings_clean.drop_duplicates(keep="first").reset_index(drop=True)
pulse_clean = pulse_clean.drop_duplicates(keep="first").reset_index(drop=True)

print("Duplicates removed:")
print(f"- meetings_clean: {meetings_dupes_removed:,}")
print(f"- pulse_clean   : {pulse_dupes_removed:,}")


## 2) Clean `meetings`: invalid durations, missing durations, and outliers

In [ ]:
meetings_clean["duration_was_imputed"] = False
meetings_clean["duration_outlier_flag"] = False

# Invalid durations: impossible values
invalid_duration_mask = meetings_clean["duration_mins"].notna() & (meetings_clean["duration_mins"] <= 0)
invalid_duration_count = int(invalid_duration_mask.sum())
if invalid_duration_count > 0:
    meetings_clean.loc[invalid_duration_mask, "duration_mins"] = np.nan
    meetings_clean.loc[invalid_duration_mask, "duration_was_imputed"] = True

# Missing durations: impute by meeting_type median, fallback to overall median
type_median_duration = meetings_clean.groupby("meeting_type")["duration_mins"].median()
overall_median_duration = float(meetings_clean["duration_mins"].median())

missing_duration_mask = meetings_clean["duration_mins"].isna()
missing_duration_count = int(missing_duration_mask.sum())

if missing_duration_count > 0:
    fill_values = meetings_clean.loc[missing_duration_mask, "meeting_type"].map(type_median_duration).fillna(overall_median_duration)
    meetings_clean.loc[missing_duration_mask, "duration_mins"] = fill_values
    meetings_clean.loc[missing_duration_mask, "duration_was_imputed"] = True

# Outliers: cap >240 mins and keep a flag for transparency
outlier_mask = meetings_clean["duration_mins"] > 240
outlier_count = int(outlier_mask.sum())
meetings_clean.loc[outlier_mask, "duration_outlier_flag"] = True
meetings_clean.loc[outlier_mask, "duration_mins"] = 240

meetings_clean["duration_mins"] = meetings_clean["duration_mins"].round(2)

# Recompute row_cost after duration fixes
meetings_clean["row_cost"] = ((meetings_clean["duration_mins"] / 60.0) * meetings_clean["hourly_rate"]).round(2)

print("Meetings duration fixes:")
print(f"- invalid duration rows corrected : {invalid_duration_count:,}")
print(f"- missing duration rows imputed   : {missing_duration_count:,}")
print(f"- outlier duration rows capped    : {outlier_count:,}")


## 3) Clean `pulse`: impute survey fields with context-aware medians

In [ ]:
pulse_clean["productivity_was_imputed"] = False
pulse_clean["burnout_was_imputed"] = False

# Create meeting-hours bucket to preserve threshold behavior during imputation
pulse_clean["hours_bucket"] = pd.cut(
    pulse_clean["hours_in_meetings"],
    bins=[-0.1, 5, 10, 15, 20, np.inf],
    labels=["0-5", "5-10", "10-15", "15-20", "20+"],
)

group_keys = ["seniority", "department", "hours_bucket"]

prod_group_median = pulse_clean.groupby(group_keys, observed=True)["self_reported_productivity"].median()
prod_global_median = float(pulse_clean["self_reported_productivity"].median())

burn_group_median = pulse_clean.groupby(group_keys, observed=True)["self_reported_burnout"].median()
burn_global_median = float(pulse_clean["self_reported_burnout"].median())

prod_missing_mask = pulse_clean["self_reported_productivity"].isna()
burn_missing_mask = pulse_clean["self_reported_burnout"].isna()

prod_imputed_count = int(prod_missing_mask.sum())
burn_imputed_count = int(burn_missing_mask.sum())

if prod_imputed_count > 0:
    prod_fill = (
        pulse_clean.loc[prod_missing_mask, group_keys]
        .apply(tuple, axis=1)
        .map(prod_group_median)
        .fillna(prod_global_median)
    )
    pulse_clean.loc[prod_missing_mask, "self_reported_productivity"] = prod_fill
    pulse_clean.loc[prod_missing_mask, "productivity_was_imputed"] = True

if burn_imputed_count > 0:
    burn_fill = (
        pulse_clean.loc[burn_missing_mask, group_keys]
        .apply(tuple, axis=1)
        .map(burn_group_median)
        .fillna(burn_global_median)
    )
    pulse_clean.loc[burn_missing_mask, "self_reported_burnout"] = burn_fill
    pulse_clean.loc[burn_missing_mask, "burnout_was_imputed"] = True

pulse_clean["self_reported_productivity"] = pulse_clean["self_reported_productivity"].round().clip(1, 10).astype(int)
pulse_clean["self_reported_burnout"] = pulse_clean["self_reported_burnout"].round().clip(1, 10).astype(int)

pulse_clean = pulse_clean.drop(columns=["hours_bucket"])

print("Pulse survey imputation:")
print(f"- productivity rows imputed: {prod_imputed_count:,}")
print(f"- burnout rows imputed     : {burn_imputed_count:,}")


## 4) Final checks, export, and before/after summary

In [ ]:
meetings_out = PROCESSED_DIR / "meetings_clean.csv"
pulse_out = PROCESSED_DIR / "pulse_clean.csv"

meetings_clean.to_csv(meetings_out, index=False)
pulse_clean.to_csv(pulse_out, index=False)

after_counts = {
    "meetings_rows": len(meetings_clean),
    "pulse_rows": len(pulse_clean),
}

print("=== AFTER CLEANING ===")
print(after_counts)

after_meet_missing = missing_summary(meetings_clean, "meetings_clean")
after_pulse_missing = missing_summary(pulse_clean, "pulse_clean")

print("\n=== EXPORT PATHS ===")
print(meetings_out.resolve())
print(pulse_out.resolve())

print("\n=== BEFORE vs AFTER ROW COUNTS ===")
row_compare = pd.DataFrame(
    {
        "before": pd.Series(before_counts),
        "after": pd.Series(after_counts),
    }
)
row_compare["delta"] = row_compare["after"] - row_compare["before"]
print(row_compare)

print("\n=== CLEANING IMPACT COUNTS ===")
impact = {
    "meetings_duplicates_removed": meetings_dupes_removed,
    "pulse_duplicates_removed": pulse_dupes_removed,
    "meetings_invalid_duration_fixed": invalid_duration_count,
    "meetings_missing_duration_imputed": missing_duration_count,
    "meetings_outlier_duration_capped": outlier_count,
    "pulse_productivity_imputed": prod_imputed_count,
    "pulse_burnout_imputed": burn_imputed_count,
}
for k, v in impact.items():
    print(f"{k}: {v:,}")
